# 🧬 Cancer Prediction Using Machine Learning

## Phase 5 — Feature Engineering

**Objective:** Select the most informative features and remove redundancy
to improve model performance and interpretability.

---

### Why Feature Engineering?

More features ≠ better model. In fact, too many features can cause:
- **Overfitting** — model memorizes noise instead of learning patterns
- **Curse of dimensionality** — distance metrics break down in high dimensions
- **Slower training** — more features = more computation
- **Harder to interpret** — which features actually matter?

In [ ]:
# Setup
import os, sys, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif

%matplotlib inline
sns.set_style('whitegrid')

src_path = os.path.abspath(os.path.join('..', 'src'))
if src_path not in sys.path:
    sys.path.insert(0, src_path)
from utils import print_section_header

# Load cleaned data
df = pd.read_csv('../data/processed/breast_cancer_cleaned.csv')
feature_cols = [col for col in df.columns if col != 'diagnosis']
X = df[feature_cols]
y = df['diagnosis']

print(f"✅ Loaded: {X.shape[0]} samples, {X.shape[1]} features")

---

## 1. 🌲 Feature Importance (Random Forest)

In [ ]:
# =============================================================================
# CELL 2: Random Forest Feature Importance
# =============================================================================
# WHY: Random Forest measures how much each feature contributes to
#       reducing impurity (Gini) across all decision trees. Features
#       that consistently appear at the top of trees are most important.
#
# NOTE: This is a PRELIMINARY importance — we train on ALL data here
#       just for feature selection. The real model will use train/test split.
# =============================================================================

print_section_header("Random Forest Feature Importance", "🌲")

# Train a quick Random Forest for feature importance
rf_temp = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1  # Use all CPU cores
)
rf_temp.fit(X, y)

# Get importance scores
importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_temp.feature_importances_
}).sort_values('Importance', ascending=False)

# Interactive bar chart
fig = px.bar(
    importance.head(15),
    x='Importance',
    y='Feature',
    orientation='h',
    title='🌲 Top 15 Features by Random Forest Importance',
    color='Importance',
    color_continuous_scale='Viridis'
)
fig.update_layout(height=500, template='plotly_dark',
                  yaxis={'categoryorder': 'total ascending'})
fig.show()

try:
    fig.write_image('../images/feature_importance_rf.png', scale=2)
    print("✅ Saved: images/feature_importance_rf.png")
except Exception as e:
    print(f"⚠️ Could not save: {e}")

print("\nTop 10 most important features:")
for i, row in importance.head(10).iterrows():
    print(f"   {row['Feature']:35s} → {row['Importance']:.4f}")

---

## 2. 🔗 Correlation-Based Feature Removal

We remove one feature from each pair where |correlation| > 0.95.
When choosing which to drop, we keep the one with higher correlation to the target.

In [ ]:
# =============================================================================
# CELL 3: Remove Highly Correlated Features
# =============================================================================
# WHY: Features with correlation > 0.95 are nearly identical.
#       Keeping both wastes computation and can destabilize
#       linear models (multicollinearity).
#
# STRATEGY: For each correlated pair, keep the one that has
#       higher correlation with the target variable.
# =============================================================================

print_section_header("Correlation-Based Feature Selection", "🔗")

corr_matrix = X.corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones_like(corr_matrix, dtype=bool), k=1))

# Target correlation for deciding which to keep
target_corr = X.corrwith(y).abs()

# Find features to drop (correlation > 0.95)
threshold = 0.95
features_to_drop = set()

for col in upper_tri.columns:
    highly_correlated = upper_tri.index[upper_tri[col] > threshold].tolist()
    for corr_feat in highly_correlated:
        # Drop the one with LOWER correlation to target
        if target_corr[col] >= target_corr[corr_feat]:
            features_to_drop.add(corr_feat)
            print(f"  Drop '{corr_feat}' (target_corr={target_corr[corr_feat]:.3f}) "
                  f"— keep '{col}' (target_corr={target_corr[col]:.3f})")
        else:
            features_to_drop.add(col)
            print(f"  Drop '{col}' (target_corr={target_corr[col]:.3f}) "
                  f"— keep '{corr_feat}' (target_corr={target_corr[corr_feat]:.3f})")

print(f"\n📊 Features to drop: {len(features_to_drop)}")
print(f"   Dropping: {sorted(features_to_drop)}")

# Apply feature removal
selected_features = [col for col in feature_cols if col not in features_to_drop]
print(f"\n✅ Features kept: {len(selected_features)} (from {len(feature_cols)})")
print(f"   Selected: {selected_features}")

In [ ]:
# =============================================================================
# CELL 4: Final Feature Set Summary
# =============================================================================

print_section_header("Final Feature Set", "✅")

# Create summary table
final_summary = pd.DataFrame({
    'Feature': selected_features,
    'RF Importance': [importance.set_index('Feature').loc[f, 'Importance'] 
                      for f in selected_features],
    'Target Correlation': [target_corr[f] for f in selected_features]
}).sort_values('RF Importance', ascending=False)

print(final_summary.to_string(index=False))
print(f"\n📊 Total features: {len(selected_features)} (reduced from {len(feature_cols)})")
print(f"   Reduction: {len(feature_cols) - len(selected_features)} features removed ({(len(feature_cols) - len(selected_features))/len(feature_cols)*100:.0f}%)")

# Save selected features for the next notebook
import json
features_path = '../data/processed/selected_features.json'
with open(features_path, 'w') as f:
    json.dump(selected_features, f, indent=2)
print(f"\n💾 Selected features saved to: {features_path}")

---

## 📋 Phase 5 Summary

### ✔ Summary

- Computed **Random Forest feature importance** — top features are concave points, perimeter, area
- Removed **highly correlated features** (correlation > 0.95) — reduced redundancy
- Final feature set saved to `data/processed/selected_features.json`

### ✔ What You Learned

| Concept | Description |
|---------|-------------|
| Feature Importance | How much each feature contributes to predictions |
| Multicollinearity Removal | Dropping redundant correlated features |
| Curse of Dimensionality | Too many features can hurt performance |

**🎯 Interview Question:** *How do you select features for a model?*
> I use a combination of: 1) Correlation analysis to remove redundant features, 2) Model-based importance (Random Forest, XGBoost), 3) Statistical tests (mutual information), 4) Domain knowledge. I never rely on a single method — I cross-validate the feature set to ensure it generalizes.

---

*Phase 5 Complete ✅*